# Variables aleatorias: ventas de una campaña de email

Una tienda online envía cada día un email promocional a **500 clientes**. Cada cliente que lo recibe puede comprar (1) o no comprar (0). Al equipo de marketing le interesa saber:

- ¿qué proporción de clientes compra tras recibir el email?
- ¿cuántas ventas podemos esperar un día cualquiera?
- ¿con qué probabilidad un día **no cubrimos el coste** de la campaña?
- ¿con qué probabilidad tendremos **más pedidos de los que el almacén puede preparar**?

En este cuaderno vamos a:

1. Simular los datos observados de la campaña.
2. Estimar la probabilidad de compra $p$ con la media muestral.
3. Usar la función de masa de probabilidad y la función de distribución para responder preguntas de negocio.
4. Ver cómo cambian los resultados cada vez que **repetimos** el proceso con datos nuevos.
5. Aumentar el tamaño de la muestra y ver qué pasa con las estimaciones.
6. Conocer la **ley de los grandes números** y el **teorema del límite central**.

> **Importante.** Los datos se simulan **sin semilla**: cada vez que ejecutes el cuaderno saldrán datos distintos, y tus resultados no coincidirán con los de tus compañeros. Es intencionado: así veremos que una estimación depende de la muestra que nos haya tocado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## 1. El caso y el modelo

**Un cliente.** Cada cliente que recibe el email compra o no compra. Lo modelamos con una variable de **Bernoulli**:

$$Y_i \sim \text{Bernoulli}(p), \qquad Y_i = \begin{cases} 1 & \text{si el cliente } i \text{ compra (probabilidad } p) \\ 0 & \text{si no compra (probabilidad } 1 - p) \end{cases}$$

**Un día.** El número de ventas de un día es la suma de los 500 resultados individuales. Si los clientes son **independientes** y todos tienen **la misma** probabilidad de compra $p$, esa suma sigue una distribución **binomial**:

$$X = Y_1 + Y_2 + \cdots + Y_{500} \sim \text{Binomial}(n = 500,\ p)$$

**Suposiciones** (si fallan, fallan los cálculos):

- **Independencia:** la decisión de un cliente no influye en la de otro.
- **Misma distribución:** todos los clientes, todos los días, tienen la misma probabilidad de compra $p$.

**Los números del negocio:**

- Cada venta deja un **margen de 15 €**.
- La campaña cuesta **450 € al día** (plataforma de email + creatividades). Para cubrir costes hacen falta al menos $450 / 15 = 30$ ventas.
- El almacén puede preparar **como máximo 45 pedidos al día** con la plantilla actual.

In [ ]:
clientes_dia = 500  # emails enviados cada día: la n de la binomial
margen_venta = 15  # € por venta
coste_dia = 450  # € por día de campaña
capacidad = 45  # pedidos que el almacén prepara al día

ventas_equilibrio = int(np.ceil(coste_dia / margen_venta))
ventas_equilibrio

## 2. Simular los datos observados

En la vida real **no conocemos $p$**: es justo lo que queremos estimar. Para simular datos tenemos que elegir un valor "real", que hace el papel de la realidad. Imagina que está escrito en un sobre cerrado: lo usamos para generar los datos y, al final, para comprobar lo bien que lo hemos estimado.

Simulamos **30 días** de campaña. Para cada día y cada cliente generamos un 0 o un 1.

In [ ]:
p_real = 0.07  # en la práctica no la conocemos

rng = np.random.default_rng()

dias = 30
compras = rng.binomial(
    n=1, p=p_real, size=(dias, clientes_dia)
)  # una Bernoulli por cliente
compras.shape

Cada fila es un día y cada columna un cliente. Así se ven los primeros días y clientes:

In [ ]:
compras[:5, :20]

Lo que normalmente llega al equipo de marketing es el **resumen diario**: cuántas ventas hubo cada día.

In [ ]:
datos = pd.DataFrame(
    {
        "dia": np.arange(1, dias + 1),
        "emails": clientes_dia,
        "ventas": compras.sum(axis=1),
    }
)
datos.head(10)

In [ ]:
datos["ventas"].describe()

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.bar(datos["dia"], datos["ventas"], color="tab:blue")
plt.axhline(
    ventas_equilibrio,
    color="tab:red",
    linestyle="--",
    label="umbral de rentabilidad (30)",
)
plt.axhline(
    capacidad, color="tab:orange", linestyle="--", label="capacidad del almacén (45)"
)
plt.xlabel("día")
plt.ylabel("ventas")
plt.title("Ventas diarias observadas")
plt.legend()
plt.show()

## 3. Estimar la probabilidad de compra $p$

Hemos observado `dias × clientes_dia` emails y sabemos cuántos acabaron en compra. La estimación natural de $p$ es la **proporción de compras**, que es lo mismo que la **media muestral** de los ceros y unos:

$$\hat{p} = \bar{y} = \frac{1}{N}\sum_{i=1}^{N} y_i = \frac{\text{ventas totales}}{\text{emails enviados}}$$

donde $N$ es el número total de clientes observados. Este estimador es el **estimador de máxima verosimilitud** de $p$: de todos los valores posibles de $p$, es el que hace más probables los datos que hemos observado.

In [ ]:
p_hat = compras.mean()
p_hat

Con el resumen diario llegamos a lo mismo: el total de ventas entre el total de emails, o la media de ventas diarias entre 500.

In [ ]:
print(datos["ventas"].sum() / datos["emails"].sum())
print(datos["ventas"].mean() / clientes_dia)

Ahora abrimos el sobre. ¿Cuánto nos hemos equivocado? (Recuerda: en la vida real **no** podríamos hacer esta comprobación.)

In [ ]:
print(f"p real:     {p_real:.4f}")
print(f"p estimada: {p_hat:.4f}")
print(f"error:      {p_hat - p_real:+.4f}")

## 4. Preguntas de negocio con la función de masa y la función de distribución

Con $\hat{p}$ ya tenemos un modelo para las ventas de un día cualquiera:

$$X \sim \text{Binomial}(500,\ \hat{p})$$

Recuerda las dos funciones que describen una variable discreta:

- **Función de masa de probabilidad** (`pmf`): $f(k) = P(X = k)$, la probabilidad de **un valor concreto**.
- **Función de distribución** (`cdf`): $F(k) = P(X \le k)$, la probabilidad **acumulada** hasta $k$. Sirve para rangos de valores.

Con la función de distribución se calculan todos los rangos:

| Pregunta | Fórmula | En Python |
|---|---|---|
| $P(X \le k)$ | $F(k)$ | `cdf(k)` |
| $P(X < k)$ | $F(k-1)$ | `cdf(k - 1)` |
| $P(X > k)$ | $1 - F(k)$ | `1 - cdf(k)` o `sf(k)` |
| $P(X \ge k)$ | $1 - F(k-1)$ | `1 - cdf(k - 1)` |
| $P(a \le X \le b)$ | $F(b) - F(a-1)$ | `cdf(b) - cdf(a - 1)` |

En una variable discreta hay que tener cuidado con los extremos: $P(X < 30)$ y $P(X \le 30)$ **no** son lo mismo.

In [ ]:
modelo = stats.binom(n=clientes_dia, p=p_hat)

print(f"Ventas esperadas al día: {modelo.mean():.1f}")
print(f"Desviación típica:        {modelo.std():.1f}")

**Pregunta 1.** ¿Cuál es la probabilidad de que un día haya **exactamente 35 ventas**?

In [ ]:
modelo.pmf(35)

Es una probabilidad pequeña, aunque 35 esté cerca de lo esperado: con tantos valores posibles, cada uno por separado es poco probable. Por eso, en el negocio casi siempre preguntamos por **rangos**.

**Pregunta 2.** ¿Cuál es la probabilidad de que un día **perdamos dinero**, es decir, de que haya menos de 30 ventas?

$$P(X < 30) = P(X \le 29) = F(29)$$

In [ ]:
p_perdidas = modelo.cdf(ventas_equilibrio - 1)
p_perdidas

La función de distribución es la suma de la función de masa. Podemos comprobarlo sumando $P(X = 0) + P(X = 1) + \cdots + P(X = 29)$:

In [ ]:
modelo.pmf(np.arange(0, ventas_equilibrio)).sum()

**Pregunta 3.** ¿Cuál es la probabilidad de que un día **se sature el almacén** (más de 45 pedidos)?

$$P(X > 45) = 1 - F(45)$$

In [ ]:
p_saturacion = 1 - modelo.cdf(capacidad)
print(p_saturacion)
print(modelo.sf(capacidad))  # sf = 1 - cdf, lo mismo

**Pregunta 4.** ¿Cuál es la probabilidad de un **día normal**: rentable y sin saturar el almacén, entre 30 y 45 ventas?

$$P(30 \le X \le 45) = F(45) - F(29)$$

In [ ]:
p_normal = modelo.cdf(capacidad) - modelo.cdf(ventas_equilibrio - 1)
p_normal

Las tres probabilidades cubren todos los casos posibles, así que deben sumar 1:

In [ ]:
p_perdidas + p_normal + p_saturacion

**Pregunta 5.** En un mes de 30 días, ¿cuántos días esperamos perder dinero? ¿Y cuánto ganamos de media al mes?

In [ ]:
print(f"Días con pérdidas al mes: {30 * p_perdidas:.1f}")
print(f"Beneficio medio al día:   {margen_venta * modelo.mean() - coste_dia:.0f} €")
print(
    f"Beneficio medio al mes:   {30 * (margen_venta * modelo.mean() - coste_dia):.0f} €"
)

### ¿Se parece el modelo a los datos?

Comparamos lo que dice el modelo con lo que hemos visto en nuestros 30 días. Con solo 30 días, las frecuencias observadas serán bastante irregulares.

In [ ]:
k = np.arange(10, 65)

plt.figure(figsize=(9, 4))
plt.hist(
    datos["ventas"],
    bins=np.arange(9.5, 65.5, 1),
    density=True,
    alpha=0.5,
    label="frecuencia observada (30 días)",
)
plt.plot(k, modelo.pmf(k), "o-", color="black", markersize=3, label="Binomial(500, p̂)")
plt.axvline(ventas_equilibrio - 0.5, color="tab:red", linestyle="--")
plt.axvline(capacidad + 0.5, color="tab:orange", linestyle="--")
plt.xlabel("ventas en un día")
plt.ylabel("probabilidad")
plt.legend()
plt.show()

In [ ]:
resumen = pd.DataFrame(
    {
        "modelo": [p_perdidas, p_normal, p_saturacion],
        "observado en 30 días": [
            (datos["ventas"] < ventas_equilibrio).mean(),
            datos["ventas"].between(ventas_equilibrio, capacidad).mean(),
            (datos["ventas"] > capacidad).mean(),
        ],
    },
    index=["pérdidas (< 30)", "normal (30-45)", "saturación (> 45)"],
)
resumen.round(3)

Las dos columnas no coinciden exactamente. La columna "observado" se basa en solo 30 días y es muy irregular. El modelo aprovecha mucho mejor la información: usa los 15 000 emails para estimar $p$ y deduce de ahí la distribución completa.

Fíjate también en la función de distribución: es una escalera que sube de 0 a 1, y la altura en cada punto es $P(X \le k)$.

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.step(k, modelo.cdf(k), where="post", label="F(k) del modelo")
plt.step(
    np.sort(datos["ventas"]),
    np.arange(1, dias + 1) / dias,
    where="post",
    label="F empírica (30 días)",
)
plt.axvline(ventas_equilibrio - 1, color="tab:red", linestyle="--")
plt.xlabel("ventas en un día (k)")
plt.ylabel("P(X ≤ k)")
plt.legend()
plt.show()

## 5. Otra muestra, otros resultados

Si la campaña hubiera durado **otros** 30 días, los datos habrían sido distintos y también $\hat{p}$ y todas las probabilidades que hemos calculado.

**Prueba:** vuelve a ejecutar el cuaderno entero (*Entorno de ejecución → Ejecutar todas*) y apunta $\hat{p}$ y la probabilidad de perder dinero. ¿Cuánto cambian?

Para no tener que hacerlo a mano, metemos todo el análisis en una función y la repetimos varias veces.

In [ ]:
def analizar_campana(dias, clientes_dia=clientes_dia, p=p_real):
    # Simula una campaña de `dias` días y devuelve las estimaciones
    compras = rng.binomial(n=1, p=p, size=(dias, clientes_dia))
    p_hat = compras.mean()
    modelo = stats.binom(n=clientes_dia, p=p_hat)
    return {
        "p_hat": p_hat,
        "ventas esperadas": modelo.mean(),
        "P(pérdidas)": modelo.cdf(ventas_equilibrio - 1),
        "P(saturación)": modelo.sf(capacidad),
    }

In [ ]:
repeticiones = pd.DataFrame([analizar_campana(dias=30) for _ in range(10)])
repeticiones.round(4)

In [ ]:
repeticiones.describe().loc[["mean", "std", "min", "max"]].round(4)

Las estimaciones de $p$ se mueven poco, pero las probabilidades de negocio se mueven bastante más, sobre todo las de las colas. Un cambio pequeño en $\hat{p}$ desplaza toda la distribución y cambia mucho la probabilidad de los valores extremos.

**Para pensar:** si tu jefe te pregunta "¿cuál es la probabilidad de perder dinero un día?", ¿basta con dar un número? ¿Qué más le dirías?

## 6. ¿Qué pasa si observamos más datos?

Ahora repetimos el experimento con campañas de distinta duración: 1 semana, 1 mes, 3 meses, 1 año y 3 años. Para cada duración simulamos **500 campañas** y miramos cómo se reparten las estimaciones de $p$.

Ojo con no confundir dos "tamaños": la $n = 500$ de la binomial (emails por día) no cambia. Lo que aumenta es el **tamaño de la muestra**: el número de días observados y, con ellos, el total de clientes $N = \text{días} \times 500$.

In [ ]:
duraciones = [7, 30, 90, 365, 1095]
estimaciones = {}
for d in duraciones:
    # cada campaña de d días: el total de compras de d·500 clientes es una Binomial(d·500, p)
    N = d * clientes_dia
    estimaciones[d] = rng.binomial(n=N, p=p_real, size=500) / N

tabla = pd.DataFrame(
    {
        "clientes observados (N)": [d * clientes_dia for d in duraciones],
        "media de p̂": [estimaciones[d].mean() for d in duraciones],
        "desv. típica de p̂": [estimaciones[d].std() for d in duraciones],
        "p̂ mínima": [estimaciones[d].min() for d in duraciones],
        "p̂ máxima": [estimaciones[d].max() for d in duraciones],
    },
    index=pd.Index(duraciones, name="días"),
)
tabla.round(4)

In [ ]:
plt.figure(figsize=(9, 4))
plt.boxplot(
    [estimaciones[d] for d in duraciones], tick_labels=[str(d) for d in duraciones]
)
plt.axhline(p_real, color="tab:red", linestyle="--", label="p real")
plt.xlabel("días observados")
plt.ylabel("p̂")
plt.title("Estimaciones de p en 500 campañas simuladas de cada duración")
plt.legend()
plt.show()

Dos cosas que conviene observar:

- Con cualquier duración, las estimaciones están **centradas en el valor real**: la media de $\hat{p}$ es prácticamente $p$. El estimador no se equivoca sistemáticamente hacia arriba ni hacia abajo.
- Al aumentar la muestra, la **dispersión baja**. Pero no proporcionalmente: para dividir la desviación típica a la mitad hace falta **cuadruplicar** los datos. Compara la desviación típica con la fórmula teórica $\sqrt{p(1-p)/N}$:

In [ ]:
tabla["fórmula √(p(1-p)/N)"] = np.sqrt(
    p_real * (1 - p_real) / tabla["clientes observados (N)"]
)
tabla[["clientes observados (N)", "desv. típica de p̂", "fórmula √(p(1-p)/N)"]].round(5)

¿Y las probabilidades de negocio? Repetimos el análisis completo con 7 días y con 365 días:

In [ ]:
comparacion = pd.concat(
    {
        "7 días": pd.DataFrame([analizar_campana(dias=7) for _ in range(200)]).std(),
        "30 días": pd.DataFrame([analizar_campana(dias=30) for _ in range(200)]).std(),
        "365 días": pd.DataFrame(
            [analizar_campana(dias=365) for _ in range(200)]
        ).std(),
    },
    axis=1,
)
print("Desviación típica de cada estimación en 200 repeticiones:")
comparacion.round(4)

## 7. La ley de los grandes números

Lo que acabamos de ver tiene nombre.

> **Ley de los grandes números.** Si $Y_1, Y_2, \ldots$ son independientes y tienen la misma distribución, con media $\mu$, la media muestral $\bar{Y}_N = \frac{1}{N}\sum_{i=1}^{N} Y_i$ se acerca cada vez más a $\mu$ a medida que $N$ crece.

En nuestro caso $\mu = p$: la proporción de compras observada se acerca a la probabilidad real de compra cuanto más clientes observamos. Es lo que justifica estimar probabilidades con frecuencias.

Para verlo, seguimos a una única campaña cliente a cliente y calculamos la proporción de compras acumulada después de cada email. Lo repetimos para 5 campañas distintas.

In [ ]:
N_max = 50_000
N = np.arange(1, N_max + 1)

plt.figure(figsize=(9, 4))
for _ in range(5):
    y = rng.binomial(n=1, p=p_real, size=N_max)
    plt.plot(N, y.cumsum() / N, linewidth=1)
plt.axhline(p_real, color="black", linestyle="--", label="p real")
plt.xscale("log")
plt.ylim(0, 0.2)
plt.xlabel("clientes observados N (escala logarítmica)")
plt.ylabel("proporción de compras acumulada")
plt.title("Ley de los grandes números: 5 campañas")
plt.legend()
plt.show()

Al principio, con pocos clientes, la proporción da saltos enormes: tras el primer cliente vale 0 o 1. Al acumular clientes, todas las trayectorias se van pegando a $p$.

**Lo que la ley no dice:** que después de una racha de días malos "toque" una racha buena para compensar. Cada cliente sigue siendo independiente. La media se acerca a $p$ porque los nuevos datos **diluyen** las rachas, no porque las compensen.

## 8. El teorema del límite central

La ley de los grandes números dice **hacia dónde** va $\hat{p}$. Pero con una muestra finita, $\hat{p}$ sigue teniendo un error. ¿Cómo se distribuye ese error? Es decir, si repitiéramos la campaña muchas veces, ¿qué forma tendría el histograma de los $\hat{p}$?

> **Teorema del límite central (TLC).** Si $Y_1, \ldots, Y_N$ son independientes, con la misma distribución de media $\mu$ y varianza $\sigma^2$, para $N$ suficientemente grande la media muestral es aproximadamente normal:
> $$\bar{Y}_N \approx N\!\left(\mu,\ \frac{\sigma^2}{N}\right)$$
> **sea cual sea la distribución de las $Y_i$.**

Cada $Y_i$ es un 0 o un 1: no se parece en nada a una campana. Para una Bernoulli, $\mu = p$ y $\sigma^2 = p(1-p)$, así que el TLC dice que

$$\hat{p} \approx N\!\left(p,\ \frac{p(1-p)}{N}\right)$$

Comprobémoslo simulando 10 000 campañas con distinto número de clientes $N$:

In [ ]:
tamanos = [20, 200, 2000, 20_000]

fig, axes = plt.subplots(1, len(tamanos), figsize=(14, 3.5))
for ax, N in zip(axes, tamanos):
    p_hats = rng.binomial(n=N, p=p_real, size=10_000) / N
    se = np.sqrt(p_real * (1 - p_real) / N)
    # en N pequeños p̂ solo toma valores k/N: un bin por cada valor posible
    bins = (
        np.arange(p_hats.min() - 0.5 / N, p_hats.max() + 1 / N, 1 / N)
        if N <= 2000
        else 40
    )
    ax.hist(p_hats, bins=bins, density=True, alpha=0.6)
    x = np.linspace(p_real - 4 * se, p_real + 4 * se, 300)
    ax.plot(x, stats.norm.pdf(x, loc=p_real, scale=se), color="black")
    ax.set_title(f"N = {N}")
    ax.set_xlabel("p̂")
axes[0].set_ylabel("densidad")
fig.suptitle("Distribución de p̂ en 10 000 campañas simuladas (línea: normal del TLC)")
plt.tight_layout()
plt.show()

Con $N = 20$ el histograma es asimétrico y muy irregular: con $p = 0.07$, lo más frecuente es que haya 0, 1 o 2 compras, y la normal predice valores negativos de $\hat{p}$, que son imposibles. Con $N = 200$ ya se parece a una campana, y a partir de ahí la aproximación es excelente.

Una regla práctica para proporciones: la aproximación normal funciona bien cuando hay al menos 5 éxitos y 5 fracasos esperados, $Np \ge 5$ y $N(1-p) \ge 5$.

In [ ]:
for N in tamanos:
    print(f"N = {N:>6}:  N·p = {N * p_real:>7.1f}")

### El TLC también explica la forma de la binomial

Las ventas de un día son una **suma** de 500 Bernoullis, y una suma es la media multiplicada por 500. Por eso el TLC dice que la binomial también es aproximadamente normal, con la misma media y desviación típica:

$$X \sim \text{Binomial}(n, p) \approx N\big(np,\ np(1-p)\big)$$

In [ ]:
mu = clientes_dia * p_hat
sigma = np.sqrt(clientes_dia * p_hat * (1 - p_hat))

plt.figure(figsize=(9, 4))
plt.bar(k, modelo.pmf(k), alpha=0.5, label="Binomial(500, p̂)")
x = np.linspace(k.min(), k.max(), 300)
plt.plot(
    x,
    stats.norm.pdf(x, loc=mu, scale=sigma),
    color="black",
    label="Normal(np̂, np̂(1-p̂))",
)
plt.xlabel("ventas en un día")
plt.ylabel("probabilidad")
plt.legend()
plt.show()

Podemos repetir la pregunta 2 (probabilidad de perder dinero) con la aproximación normal. Como la binomial es discreta y la normal continua, se usa la **corrección por continuidad**: "29 o menos" en la binomial equivale a "menos de 29.5" en la normal.

In [ ]:
print(f"Binomial exacta:          {modelo.cdf(29):.4f}")
print(f"Normal sin corrección:    {stats.norm.cdf(29, loc=mu, scale=sigma):.4f}")
print(f"Normal con corrección:    {stats.norm.cdf(29.5, loc=mu, scale=sigma):.4f}")

### Para qué sirve el TLC: cuantificar el error de la estimación

En la vida real solo tenemos **una** campaña y no conocemos $p$. El TLC nos dice cuánto puede equivocarse $\hat{p}$ sin tener que repetir la campaña: el 95 % de las veces, $\hat{p}$ cae a menos de $1.96$ desviaciones típicas de $p$. La desviación típica de $\hat{p}$ se llama **error estándar** y la estimamos con los propios datos:

$$\text{EE} = \sqrt{\frac{\hat{p}(1-\hat{p})}{N}}$$

In [ ]:
N_obs = compras.size  # clientes observados en nuestros 30 días
ee = np.sqrt(p_hat * (1 - p_hat) / N_obs)
z = stats.norm.ppf(0.975)  # 1.96

print(f"p̂ = {p_hat:.4f},  error estándar = {ee:.4f}")
print(f"Intervalo de confianza del 95 %: [{p_hat - z * ee:.4f}, {p_hat + z * ee:.4f}]")
print(f"¿Contiene el p real ({p_real})?  {p_hat - z * ee <= p_real <= p_hat + z * ee}")

Si ejecutas el cuaderno muchas veces, el intervalo contendrá el valor real aproximadamente en el 95 % de las ejecuciones. Lo comprobamos simulando 1000 campañas de 30 días:

In [ ]:
N_obs = 30 * clientes_dia
p_hats = rng.binomial(n=N_obs, p=p_real, size=1000) / N_obs
ees = np.sqrt(p_hats * (1 - p_hats) / N_obs)
contiene = (p_hats - z * ees <= p_real) & (p_real <= p_hats + z * ees)
contiene.mean()

## Resumen

- Cada cliente es una **Bernoulli($p$)** y las ventas de un día, una **Binomial($n$, $p$)**, siempre que los clientes sean independientes y tengan la misma probabilidad de compra.
- La **media muestral** $\hat{p}$ (la proporción de compras) es el **estimador de máxima verosimilitud** de $p$.
- La **función de masa** (`pmf`) da la probabilidad de un valor concreto. La **función de distribución** (`cdf`) da probabilidades acumuladas y, con ella, la de cualquier rango.
- Con otra muestra, $\hat{p}$ y todas las probabilidades calculadas con ella cambian: **una estimación es una variable aleatoria**.
- Con más datos, la estimación varía menos: su desviación típica es $\sqrt{p(1-p)/N}$, que baja con la raíz de $N$.
- **Ley de los grandes números:** la media muestral se acerca a la media real al crecer $N$.
- **Teorema del límite central:** la media muestral es aproximadamente normal, aunque los datos sean ceros y unos. Eso nos permite cuantificar el error de una estimación con una sola muestra.

## Para seguir explorando

1. Cambia `p_real` a 0.02 y vuelve a ejecutar todo. ¿Qué pasa con la probabilidad de perder dinero? ¿Y con la aproximación normal para $N = 200$?
2. ¿Cuántos emails al día habría que enviar para que la probabilidad de perder dinero fuera menor del 5 %? (Ten en cuenta que el coste de la campaña no cambia.)
3. El almacén quiere ampliar su capacidad para que solo se sature un día de cada 100. ¿Cuántos pedidos al día debe poder preparar? (Pista: `modelo.ppf(0.99)`.)
4. Supón que los lunes la probabilidad de compra es el doble que el resto de días. ¿Qué suposición del modelo falla? Simúlalo y mira si la binomial sigue describiendo bien las ventas diarias.